In [1]:
import pandas as pd

data_path = "/home/jovyan/work/transformer_chatbot/data/ChatbotData.csv"
data = pd.read_csv(data_path)

data.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [2]:
print("데이터 크기:", data.shape)
print("\n결측치 개수:")
print(data.isnull().sum())

데이터 크기: (11823, 3)

결측치 개수:
Q        0
A        0
label    0
dtype: int64


In [3]:
print("전체 중복 행 개수:", data.duplicated().sum())
print("질문(Q) 중복 개수:", data["Q"].duplicated().sum())
print("답변(A) 중복 개수:", data["A"].duplicated().sum())

전체 중복 행 개수: 0
질문(Q) 중복 개수: 161
답변(A) 중복 개수: 4044


In [4]:
import re

def preprocess_sentence(sentence):
    sentence = str(sentence).strip()

    # 한글, 영문, 숫자, 기본 문장부호를 제외한 특수문자 제거
    sentence = re.sub(r"[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9?.!,]+", " ", sentence)

    # 문장부호 앞뒤에 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)

    # 여러 개의 공백을 하나로 정리
    sentence = re.sub(r"\s+", " ", sentence)

    return sentence.strip()

In [5]:
questions = data["Q"].apply(preprocess_sentence).tolist()
answers = data["A"].apply(preprocess_sentence).tolist()

print("질문:", questions[:5])
print("답변:", answers[:5])
print("질문 개수:", len(questions))
print("답변 개수:", len(answers))

질문: ['12시 땡 !', '1지망 학교 떨어졌어', '3박4일 놀러가고 싶다', '3박4일 정도 놀러가고 싶다', 'PPL 심하네']
답변: ['하루가 또 가네요 .', '위로해 드립니다 .', '여행은 언제나 좋죠 .', '여행은 언제나 좋죠 .', '눈살이 찌푸려지죠 .']
질문 개수: 11823
답변 개수: 11823


In [6]:
import sentencepiece as spm

print("SentencePiece 사용 가능!")

ModuleNotFoundError: No module named 'sentencepiece'

In [7]:
!pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 8.5 MB/s eta 0:00:00-:--:--


In [8]:
import sentencepiece as spm

print("SentencePiece 사용 가능!")

SentencePiece 사용 가능!


In [9]:
with open("chatbot_corpus.txt", "w", encoding="utf-8") as f:
    for q, a in zip(questions, answers):
        f.write(q + "\n")
        f.write(a + "\n")

spm.SentencePieceTrainer.train(
    input="chatbot_corpus.txt",
    model_prefix="chatbot_spm",
    vocab_size=8000,
    model_type="bpe",
    character_coverage=1.0
)

print("SentencePiece 모델 학습 완료!")

SentencePiece 모델 학습 완료!


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot_corpus.txt
  input_format: 
  model_prefix: chatbot_spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  diffe

In [10]:
sp = spm.SentencePieceProcessor()
sp.load("chatbot_spm.model")

test_sentence = "오늘 기분이 너무 좋아"

tokens = sp.encode_as_pieces(test_sentence)
ids = sp.encode_as_ids(test_sentence)

print("토큰:", tokens)
print("토큰 ID:", ids)
print("복원:", sp.decode_ids(ids))
print("단어장 크기:", sp.get_piece_size())

토큰: ['▁오늘', '▁기분이', '▁너무', '▁좋아']
토큰 ID: [128, 938, 55, 19]
복원: 오늘 기분이 너무 좋아
단어장 크기: 8000


In [12]:
import torch

START_TOKEN = sp.bos_id()   # 1
END_TOKEN = sp.eos_id()     # 2
PAD_TOKEN = sp.get_piece_size()   # 8000
MAX_LENGTH = 40

def encode_sentence(sentence):
    token_ids = sp.encode_as_ids(sentence)
    token_ids = [START_TOKEN] + token_ids + [END_TOKEN]

    # 너무 길면 자르고, 마지막은 종료 토큰으로 유지
    if len(token_ids) > MAX_LENGTH:
        token_ids = token_ids[:MAX_LENGTH]
        token_ids[-1] = END_TOKEN

    # 짧으면 뒤에 패딩 추가
    token_ids += [PAD_TOKEN] * (MAX_LENGTH - len(token_ids))

    return token_ids

question_ids = torch.tensor(
    [encode_sentence(q) for q in questions],
    dtype=torch.long
)

answer_ids = torch.tensor(
    [encode_sentence(a) for a in answers],
    dtype=torch.long
)

print("질문 shape:", question_ids.shape)
print("답변 shape:", answer_ids.shape)
print("첫 질문:", question_ids[0])
print("첫 답변:", answer_ids[0])
print("패딩 토큰:", PAD_TOKEN)

질문 shape: torch.Size([11823, 40])
답변 shape: torch.Size([11823, 40])
첫 질문: tensor([   1, 5559, 6807, 3206,  107,    2, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000])
첫 답변: tensor([   1, 4491,  213, 5929,    3,    2, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000])
패딩 토큰: 8000


In [13]:
decoder_input = answer_ids[:, :-1]
decoder_target = answer_ids[:, 1:]

print("디코더 입력 shape:", decoder_input.shape)
print("디코더 정답 shape:", decoder_target.shape)

print("첫 디코더 입력:", decoder_input[0])
print("첫 디코더 정답:", decoder_target[0])

디코더 입력 shape: torch.Size([11823, 39])
디코더 정답 shape: torch.Size([11823, 39])
첫 디코더 입력: tensor([   1, 4491,  213, 5929,    3,    2, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000])
첫 디코더 정답: tensor([4491,  213, 5929,    3,    2, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000, 8000,
        8000, 8000, 8000])


In [14]:
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(
    question_ids,
    decoder_input,
    decoder_target
)

train_size = int(len(dataset) * 0.9)
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("전체 데이터:", len(dataset))
print("학습 데이터:", len(train_dataset))
print("검증 데이터:", len(val_dataset))
print("학습 배치 개수:", len(train_loader))

src_batch, tgt_input_batch, tgt_target_batch = next(iter(train_loader))

print("질문 배치 shape:", src_batch.shape)
print("디코더 입력 배치 shape:", tgt_input_batch.shape)
print("디코더 정답 배치 shape:", tgt_target_batch.shape)

전체 데이터: 11823
학습 데이터: 10640
검증 데이터: 1183
학습 배치 개수: 167
질문 배치 shape: torch.Size([64, 40])
디코더 입력 배치 shape: torch.Size([64, 39])
디코더 정답 배치 shape: torch.Size([64, 39])


In [16]:
import math
import torch.nn as nn

VOCAB_SIZE = sp.get_piece_size() + 1   # SentencePiece 8000개 + PAD 1개
D_MODEL = 256
NHEAD = 8
NUM_LAYERS = 3
DIM_FEEDFORWARD = 512
DROPOUT = 0.1

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerChatbot(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_TOKEN
        )

        self.positional_encoding = PositionalEncoding(D_MODEL)

        self.transformer = nn.Transformer(
            d_model=D_MODEL,
            nhead=NHEAD,
            num_encoder_layers=NUM_LAYERS,
            num_decoder_layers=NUM_LAYERS,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=DROPOUT,
            batch_first=True
        )

        self.output_layer = nn.Linear(D_MODEL, VOCAB_SIZE)

    def generate_square_subsequent_mask(self, size, device):
        mask = torch.triu(
            torch.ones(size, size, device=device),
            diagonal=1
        )
        mask = mask.masked_fill(mask == 1, float("-inf"))
        return mask

    def forward(self, src, tgt):
        src_padding_mask = src.eq(PAD_TOKEN)
        tgt_padding_mask = tgt.eq(PAD_TOKEN)

        tgt_mask = self.generate_square_subsequent_mask(
            tgt.size(1),
            tgt.device
        )

        src_emb = self.embedding(src) * math.sqrt(D_MODEL)
        tgt_emb = self.embedding(tgt) * math.sqrt(D_MODEL)

        src_emb = self.positional_encoding(src_emb)
        tgt_emb = self.positional_encoding(tgt_emb)

        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )

        return self.output_layer(output)

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerChatbot().to(device)

print("사용 장치:", device)
print("모델 생성 완료!")
print(model)

사용 장치: cuda
모델 생성 완료!
TransformerChatbot(
  (embedding): Embedding(8001, 256, padding_idx=8000)
  (positional_encoding): PositionalEncoding()
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): Transfor

In [18]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

print("손실 함수와 옵티마이저 준비 완료!")

손실 함수와 옵티마이저 준비 완료!


In [19]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for src, tgt_input, tgt_target in dataloader:
        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_target = tgt_target.to(device)

        optimizer.zero_grad()

        output = model(src, tgt_input)

        loss = criterion(
            output.reshape(-1, VOCAB_SIZE),
            tgt_target.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [20]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt_input, tgt_target in dataloader:
            src = src.to(device)
            tgt_input = tgt_input.to(device)
            tgt_target = tgt_target.to(device)

            output = model(src, tgt_input)

            loss = criterion(
                output.reshape(-1, VOCAB_SIZE),
                tgt_target.reshape(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [21]:
EPOCHS = 10

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

/opt/conda/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
/opt/conda/lib/python3.12/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/10 | Train Loss: 6.5384 | Val Loss: 5.9664
Epoch 2/10 | Train Loss: 5.7974 | Val Loss: 5.7337
Epoch 3/10 | Train Loss: 5.5877 | Val Loss: 5.5864
Epoch 4/10 | Train Loss: 5.4051 | Val Loss: 5.4683
Epoch 5/10 | Train Loss: 5.2377 | Val Loss: 5.3678
Epoch 6/10 | Train Loss: 5.0833 | Val Loss: 5.2763
Epoch 7/10 | Train Loss: 4.9344 | Val Loss: 5.1943
Epoch 8/10 | Train Loss: 4.7922 | Val Loss: 5.1187
Epoch 9/10 | Train Loss: 4.6544 | Val Loss: 5.0517
Epoch 10/10 | Train Loss: 4.5166 | Val Loss: 4.9813


In [22]:
def generate_answer(model, sentence, sp, device, max_length=40):
    model.eval()

    # 입력 문장 전처리
    sentence = preprocess_sentence(sentence)

    # 질문을 토큰 ID로 변환
    src_ids = encode_sentence(sentence)
    src = torch.tensor([src_ids], dtype=torch.long).to(device)

    # 디코더는 시작 토큰 하나로 시작
    generated = [START_TOKEN]

    with torch.no_grad():
        for _ in range(max_length - 1):
            tgt = torch.tensor(
                [generated],
                dtype=torch.long
            ).to(device)

            output = model(src, tgt)

            # 가장 마지막 위치의 예측값
            next_token_logits = output[:, -1, :]

            # 가장 확률이 높은 토큰 선택
            next_token = next_token_logits.argmax(dim=-1).item()

            # 종료 토큰이면 생성 멈춤
            if next_token == END_TOKEN:
                break

            # 패딩 토큰은 추가하지 않음
            if next_token != PAD_TOKEN:
                generated.append(next_token)

    # 시작 토큰 제외 후 문장으로 복원
    answer_ids = generated[1:]
    answer = sp.decode_ids(answer_ids)

    return answer

In [23]:
test_questions = [
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼"
]

for question in test_questions:
    answer = generate_answer(
        model,
        question,
        sp,
        device
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 오늘 너무 힘들어
답변: 잘 될 거예요 .
------------------------------
질문: 여행 가고 싶어
답변: 잘 될 거예요 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 잘 찾아보세요 .
------------------------------


In [24]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

best_val_loss = float("inf")
early_stop_count = 0
PATIENCE = 4
ADDITIONAL_EPOCHS = 15

for epoch in range(ADDITIONAL_EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_loss)

    print(
        f"추가 Epoch {epoch + 1}/{ADDITIONAL_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # 검증 성능이 가장 좋을 때 모델 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_count = 0

        torch.save(
            model.state_dict(),
            "best_transformer_chatbot.pt"
        )

        print("→ 가장 좋은 모델 저장!")
    else:
        early_stop_count += 1

    # 검증 손실이 4번 연속 좋아지지 않으면 중단
    if early_stop_count >= PATIENCE:
        print("조기 종료: 검증 성능이 더 이상 좋아지지 않습니다.")
        break

추가 Epoch 1/15 | Train Loss: 4.3786 | Val Loss: 4.9257
→ 가장 좋은 모델 저장!
추가 Epoch 2/15 | Train Loss: 4.2451 | Val Loss: 4.8679
→ 가장 좋은 모델 저장!
추가 Epoch 3/15 | Train Loss: 4.1120 | Val Loss: 4.7987
→ 가장 좋은 모델 저장!
추가 Epoch 4/15 | Train Loss: 3.9793 | Val Loss: 4.7578
→ 가장 좋은 모델 저장!
추가 Epoch 5/15 | Train Loss: 3.8482 | Val Loss: 4.6992
→ 가장 좋은 모델 저장!
추가 Epoch 6/15 | Train Loss: 3.7142 | Val Loss: 4.6648
→ 가장 좋은 모델 저장!
추가 Epoch 7/15 | Train Loss: 3.5797 | Val Loss: 4.6116
→ 가장 좋은 모델 저장!
추가 Epoch 8/15 | Train Loss: 3.4445 | Val Loss: 4.5660
→ 가장 좋은 모델 저장!
추가 Epoch 9/15 | Train Loss: 3.3171 | Val Loss: 4.5230
→ 가장 좋은 모델 저장!
추가 Epoch 10/15 | Train Loss: 3.1816 | Val Loss: 4.4960
→ 가장 좋은 모델 저장!
추가 Epoch 11/15 | Train Loss: 3.0569 | Val Loss: 4.4729
→ 가장 좋은 모델 저장!
추가 Epoch 12/15 | Train Loss: 2.9275 | Val Loss: 4.4298
→ 가장 좋은 모델 저장!
추가 Epoch 13/15 | Train Loss: 2.8001 | Val Loss: 4.4049
→ 가장 좋은 모델 저장!
추가 Epoch 14/15 | Train Loss: 2.6795 | Val Loss: 4.3760
→ 가장 좋은 모델 저장!
추가 Epoch 15/15 | Train Loss: 

In [25]:
model.load_state_dict(
    torch.load(
        "best_transformer_chatbot.pt",
        map_location=device
    )
)

model.eval()

print("가장 성능이 좋았던 모델 불러오기 완료!")

가장 성능이 좋았던 모델 불러오기 완료!


In [26]:
test_questions = [
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "친구랑 싸웠어",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer(
        model,
        question,
        sp,
        device
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 오늘 너무 힘들어
답변: 이제 좀 괜찮아졌길 바랍니다 .
------------------------------
질문: 여행 가고 싶어
답변: 좋은 생각이에요 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 혼자도 함께도 잘 지내는 게 좋죠 .
------------------------------
질문: 친구랑 싸웠어
답변: 좋은 생각이에요 .
------------------------------
질문: 기분이 좋아
답변: 맛있게 드세요 .
------------------------------


In [27]:
def generate_answer_topk(
    model,
    sentence,
    sp,
    device,
    max_length=40,
    top_k=5,
    temperature=0.8
):
    model.eval()

    sentence = preprocess_sentence(sentence)
    src_ids = encode_sentence(sentence)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long
    ).to(device)

    generated = [START_TOKEN]

    with torch.no_grad():
        for _ in range(max_length - 1):
            tgt = torch.tensor(
                [generated],
                dtype=torch.long
            ).to(device)

            output = model(src, tgt)
            logits = output[:, -1, :] / temperature

            topk_logits, topk_indices = torch.topk(
                logits,
                k=top_k,
                dim=-1
            )

            probabilities = torch.softmax(
                topk_logits,
                dim=-1
            )

            sampled_index = torch.multinomial(
                probabilities,
                num_samples=1
            )

            next_token = topk_indices.gather(
                -1,
                sampled_index
            ).item()

            if next_token == END_TOKEN:
                break

            if next_token != PAD_TOKEN:
                generated.append(next_token)

    answer_ids = generated[1:]
    return sp.decode_ids(answer_ids)

In [28]:
test_questions = [
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "친구랑 싸웠어",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer_topk(
        model,
        question,
        sp,
        device,
        top_k=5,
        temperature=0.8
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 오늘 너무 힘들어
답변: 자신을 더 사랑해주세요 .
------------------------------
질문: 여행 가고 싶어
답변: 좋은 여행이에요 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 소원을 비세요 .
------------------------------
질문: 친구랑 싸웠어
답변: 마음 고생 많았어요 .
------------------------------
질문: 기분이 좋아
답변: 제가 있잖아요 .
------------------------------


In [ ]:
## 답변 생성 방식 비교
- Greedy Search(argmax): 안정적이지만 반복적인 답변이 많음
- Top-k Sampling: 답변은 다양하지만 일부 질문에서 맥락이 어긋남
- 최종 제출에는 Greedy Search 결과를 사용함

In [30]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

# 답변별 등장 횟수 계산
answer_counts = Counter(answers)

# train_dataset에 실제로 포함된 원본 데이터 인덱스
train_indices = train_dataset.indices

# 자주 나오는 답변은 가중치를 낮춤
# 1 / 빈도 대신 1 / sqrt(빈도)를 써서 너무 과격하게 조정하지 않음
sample_weights = [
    1.0 / (answer_counts[answers[idx]] ** 0.5)
    for idx in train_indices
]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler
)

print("균형 조정 학습 배치 개수:", len(balanced_train_loader))

균형 조정 학습 배치 개수: 167


In [31]:
best_val_loss = float("inf")

for epoch in range(8):
    train_loss = train_one_epoch(
        model,
        balanced_train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    print(
        f"균형 학습 {epoch + 1}/8 | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            model.state_dict(),
            "balanced_transformer_chatbot.pt"
        )
        print("→ 현재 최고 모델 저장")

균형 학습 1/8 | Train Loss: 2.5045 | Val Loss: 4.3557
→ 현재 최고 모델 저장
균형 학습 2/8 | Train Loss: 2.4017 | Val Loss: 4.3408
→ 현재 최고 모델 저장
균형 학습 3/8 | Train Loss: 2.2732 | Val Loss: 4.3351
→ 현재 최고 모델 저장
균형 학습 4/8 | Train Loss: 2.1555 | Val Loss: 4.3080
→ 현재 최고 모델 저장
균형 학습 5/8 | Train Loss: 2.0551 | Val Loss: 4.3026
→ 현재 최고 모델 저장
균형 학습 6/8 | Train Loss: 1.9620 | Val Loss: 4.2892
→ 현재 최고 모델 저장
균형 학습 7/8 | Train Loss: 1.8376 | Val Loss: 4.2862
→ 현재 최고 모델 저장
균형 학습 8/8 | Train Loss: 1.7435 | Val Loss: 4.2663
→ 현재 최고 모델 저장


In [32]:
model.load_state_dict(
    torch.load(
        "balanced_transformer_chatbot.pt",
        map_location=device
    )
)

model.eval()

print("균형 학습 최고 모델 불러오기 완료!")

균형 학습 최고 모델 불러오기 완료!


In [33]:
test_questions = [
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "친구랑 싸웠어",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer(
        model,
        question,
        sp,
        device
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 오늘 너무 힘들어
답변: 쉬운 일이 아니죠 .
------------------------------
질문: 여행 가고 싶어
답변: 먼저 다가가 보세요 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 철밥통 되기가 어디 쉽겠어요 .
------------------------------
질문: 친구랑 싸웠어
답변: 제가 들어드릴게요 .
------------------------------
질문: 기분이 좋아
답변: 아무런 감정들을 ?
------------------------------


In [34]:
model.load_state_dict(
    torch.load(
        "best_transformer_chatbot.pt",
        map_location=device
    )
)

model.eval()

print("이전 최고 모델로 복원 완료!")

이전 최고 모델로 복원 완료!


In [35]:
def generate_answer_beam(
    model,
    sentence,
    sp,
    device,
    max_length=40,
    beam_size=3
):
    model.eval()

    sentence = preprocess_sentence(sentence)
    src_ids = encode_sentence(sentence)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    # 각 후보: (생성된 토큰 목록, 누적 점수, 종료 여부)
    beams = [([START_TOKEN], 0.0, False)]

    with torch.no_grad():
        for _ in range(max_length - 1):
            candidates = []

            for tokens, score, finished in beams:
                if finished:
                    candidates.append((tokens, score, True))
                    continue

                tgt = torch.tensor(
                    [tokens],
                    dtype=torch.long,
                    device=device
                )

                output = model(src, tgt)
                logits = output[:, -1, :]
                log_probs = torch.log_softmax(logits, dim=-1)

                top_scores, top_indices = torch.topk(
                    log_probs,
                    k=beam_size,
                    dim=-1
                )

                for token_score, token_id in zip(
                    top_scores[0],
                    top_indices[0]
                ):
                    next_token = token_id.item()
                    new_score = score + token_score.item()

                    # PAD 토큰은 후보에서 제외
                    if next_token == PAD_TOKEN:
                        continue

                    new_tokens = tokens + [next_token]
                    is_finished = next_token == END_TOKEN

                    candidates.append(
                        (new_tokens, new_score, is_finished)
                    )

            # 길이가 긴 문장이 무조건 불리해지지 않도록 점수 보정
            candidates.sort(
                key=lambda x: x[1] / (len(x[0]) ** 0.7),
                reverse=True
            )

            beams = candidates[:beam_size]

            if all(finished for _, _, finished in beams):
                break

    best_tokens = beams[0][0]

    # START와 END 제거
    answer_ids = [
        token
        for token in best_tokens
        if token not in [START_TOKEN, END_TOKEN]
    ]

    return sp.decode_ids(answer_ids)

In [36]:
test_questions = [
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "친구랑 싸웠어",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer_beam(
        model,
        question,
        sp,
        device,
        beam_size=3
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 오늘 너무 힘들어
답변: 금방 지나갈 거예요 .
------------------------------
질문: 여행 가고 싶어
답변: 그게 최고죠 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 혼자도 함께도 잘 지내는 게 좋죠 .
------------------------------
질문: 친구랑 싸웠어
답변: 먼저 다가가 보세요 .
------------------------------
질문: 기분이 좋아
답변: 회원정보 찾기를 해보세요 .
------------------------------


In [37]:
from collections import defaultdict

MAX_SAME_ANSWER = 20
answer_seen = defaultdict(int)

filtered_questions = []
filtered_answers = []

for q, a in zip(questions, answers):
    if answer_seen[a] < MAX_SAME_ANSWER:
        filtered_questions.append(q)
        filtered_answers.append(a)
        answer_seen[a] += 1

print("기존 데이터:", len(questions))
print("정리 후 데이터:", len(filtered_questions))

기존 데이터: 11823
정리 후 데이터: 11821


In [ ]:
## 중복 답변 제한 실험

동일 답변을 최대 20개까지만 남기도록 필터링했으나,
전체 데이터가 11,823개에서 11,821개로 2개만 감소했다.

따라서 반복 답변 문제는 일부 답변이 과도하게 반복된 구조라기보다,
여러 답변이 소수씩 반복된 구조에 가까웠다.
이 필터링은 데이터 구성에 거의 영향을 주지 않아 재학습에는 사용하지 않았다.

In [ ]:
## 최종 결과

- 한국어 질문/답변 데이터 11,823개를 불러왔다.
- 결측치와 중복 여부를 확인했다.
- 정규표현식으로 한국어 문장을 전처리했다.
- SentencePiece BPE 토크나이저를 학습했다.
- 시작 토큰, 종료 토큰, 패딩 토큰을 포함한 병렬 데이터를 구성했다.
- PyTorch의 `nn.Transformer`로 한국어 챗봇 모델을 구현했다.
- 학습 손실과 검증 손실이 감소하여 모델이 정상적으로 수렴했다.
- Greedy Search, Top-k Sampling, Beam Search를 비교했다.
- 반복 답변 완화를 위해 균형 샘플링과 중복 답변 제한도 실험했다.
- 최종적으로는 추가 학습한 Greedy Search 모델이 가장 안정적인 결과를 보였다.

## 한계

- 일부 질문에서 맥락과 맞지 않는 답변이 생성되었다.
- 동일하거나 비슷한 답변이 반복되는 경향이 있었다.
- 생성 방식만 바꾸는 것보다 데이터 품질과 사전학습 모델 활용이 더 중요하다는 점을 확인했다.

In [38]:
test_question = "친구랑 싸웠어"

settings = [
    (0.5, 3),
    (0.7, 3),
    (0.7, 5),
    (0.9, 5),
]

for temperature, top_k in settings:
    print(f"\nTemperature={temperature}, Top-k={top_k}")

    for i in range(5):
        answer = generate_answer_topk(
            model,
            test_question,
            sp,
            device,
            top_k=top_k,
            temperature=temperature
        )

        print(f"{i + 1}. {answer}")


Temperature=0.5, Top-k=3
1. 좋은 결과 있을 거예요 .
2. 좋은 생각이에요 .
3. 좋은 생각이에요 .
4. 좋은 생각만 하세요 .
5. 좋은 생각만 하세요 .

Temperature=0.7, Top-k=3
1. 마음 단단히 잡으세요 .
2. 마음 단단히 잡으세요 .
3. 좋은 결과 있을 거예요 .
4. 좋은 결과 있을 거예요 .
5. 좋은 결과 있을 거예요 .

Temperature=0.7, Top-k=5
1. 좋은 생각만 하세요 .
2. 마음 단단히 잡으세요 .
3. 먼저 사과해보세요 .
4. 제가 들어드릴게요 .
5. 제가 생각해도 이건예요 .

Temperature=0.9, Top-k=5
1. 좋은 결과 있을 거예요 .
2. 좋은 생각만 하세요 .
3. 좋은 선택 하셨네요 .
4. 마음 고생 많았어요 .
5. 좋은 선택 바랍니다 .


In [39]:
test_question = "친구랑 싸웠어"

settings = [
    (0.5, 3),
    (0.7, 3),
    (0.7, 5),
    (0.9, 5),
]

for temperature, top_k in settings:
    print(f"\nTemperature={temperature}, Top-k={top_k}")

    for i in range(5):
        answer = generate_answer_topk(
            model,
            test_question,
            sp,
            device,
            top_k=top_k,
            temperature=temperature
        )

        print(f"{i + 1}. {answer}")


Temperature=0.5, Top-k=3
1. 먼저 다가가 보세요 .
2. 좋은 생각이에요 .
3. 마음 고생 많았어요 .
4. 좋은 결과 있을 거예요 .
5. 좋은 생각만 하세요 .

Temperature=0.7, Top-k=3
1. 좋은 결과 있을 거예요 .
2. 마음 고생 많았어요 .
3. 좋은 결과 있을 거예요 .
4. 좋은 생각이에요 .
5. 먼저 연락을 해보는 건 어떨까요 ?

Temperature=0.7, Top-k=5
1. 제가 들어드릴게요 .
2. 마음 단단히 잡으세요 .
3. 그 사람을 변덕스럽게이죠 .
4. 좋은 생각이에요 .
5. 좋은 선택 하셨네요 .

Temperature=0.9, Top-k=5
1. 좋은 결과 있을 거예요 .
2. 좋은 생각만 하세요 .
3. 좋은 생각이에요 .
4. 좋은 생각이에요 .
5. 먼저 다가가 보세요 .


In [40]:
test_questions = [
    "친구랑 싸웠어",
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer_improved(
        model,
        question,
        sp,
        device
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

NameError: name 'generate_answer_improved' is not defined

In [41]:
def generate_answer_improved(
    model,
    sentence,
    sp,
    device,
    max_length=40,
    top_k=3,
    temperature=0.7,
    repetition_penalty=1.2
):
    model.eval()

    sentence = preprocess_sentence(sentence)
    src_ids = encode_sentence(sentence)

    src = torch.tensor(
        [src_ids],
        dtype=torch.long,
        device=device
    )

    generated = [START_TOKEN]

    with torch.no_grad():
        for _ in range(max_length - 1):
            tgt = torch.tensor(
                [generated],
                dtype=torch.long,
                device=device
            )

            output = model(src, tgt)
            logits = output[:, -1, :] / temperature

            for token_id in set(generated):
                logits[:, token_id] /= repetition_penalty

            logits[:, PAD_TOKEN] = float("-inf")

            topk_logits, topk_indices = torch.topk(
                logits,
                k=top_k,
                dim=-1
            )

            probabilities = torch.softmax(
                topk_logits,
                dim=-1
            )

            sampled_index = torch.multinomial(
                probabilities,
                num_samples=1
            )

            next_token = topk_indices.gather(
                -1,
                sampled_index
            ).item()

            if next_token == END_TOKEN:
                break

            generated.append(next_token)

    answer_ids = generated[1:]
    return sp.decode_ids(answer_ids)

In [42]:
test_questions = [
    "친구랑 싸웠어",
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "기분이 좋아"
]

for question in test_questions:
    answer = generate_answer_improved(
        model,
        question,
        sp,
        device
    )

    print("질문:", question)
    print("답변:", answer)
    print("-" * 30)

질문: 친구랑 싸웠어
답변: 마음 단단히 잡으세요 .
------------------------------
질문: 오늘 너무 힘들어
답변: 이제 정말 안녕 .
------------------------------
질문: 여행 가고 싶어
답변: 그게 최고죠 .
------------------------------
질문: 내일 시험인데 걱정돼
답변: 공부하러 왔나봐요 .
------------------------------
질문: 기분이 좋아
답변: 제가 들어드릴게요 .
------------------------------


In [43]:
top_k=3
temperature=0.7
repetition_penalty=1.2

In [ ]:
## 생성 방식 개선 결과

- Temperature와 Top-k를 함께 조절했다.
- 낮은 Temperature는 반복적인 답변이 많았고,
  높은 Temperature는 문맥이 흔들렸다.
- `temperature=0.7`, `top_k=3`이 가장 무난했다.
- 여기에 `repetition_penalty=1.2`를 추가하자
  같은 표현이 반복되는 현상이 줄었다.
- 다만 일부 질문에서는 여전히 문맥과 맞지 않는 답변이 생성되었다.

In [44]:
import importlib.util

print("transformers:", importlib.util.find_spec("transformers"))
print("sentence_transformers:", importlib.util.find_spec("sentence_transformers"))

transformers: None
sentence_transformers: None


In [45]:
!pip install -q sentence-transformers

In [46]:
from sentence_transformers import SentenceTransformer

print("sentence-transformers 준비 완료!")

sentence-transformers 준비 완료!


In [47]:
from sentence_transformers import SentenceTransformer, util

reranker = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("한국어 문장 유사도 모델 준비 완료!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

한국어 문장 유사도 모델 준비 완료!


In [48]:
def generate_answer_rerank(
    model,
    sentence,
    sp,
    device,
    reranker,
    num_candidates=10,
    top_k=5,
    temperature=0.8,
    repetition_penalty=1.2
):
    candidates = []

    # 답변 후보 여러 개 생성
    for _ in range(num_candidates):
        answer = generate_answer_improved(
            model,
            sentence,
            sp,
            device,
            top_k=top_k,
            temperature=temperature,
            repetition_penalty=repetition_penalty
        )

        if answer and answer not in candidates:
            candidates.append(answer)

    # 후보가 하나도 없으면 빈 문자열 반환
    if not candidates:
        return "", []

    # 질문과 답변 후보를 벡터로 변환
    question_embedding = reranker.encode(
        sentence,
        convert_to_tensor=True
    )

    candidate_embeddings = reranker.encode(
        candidates,
        convert_to_tensor=True
    )

    # 질문과 각 후보의 코사인 유사도 계산
    scores = util.cos_sim(
        question_embedding,
        candidate_embeddings
    )[0]

    # 점수가 가장 높은 답변 선택
    best_index = scores.argmax().item()
    best_answer = candidates[best_index]

    # 확인용 결과
    ranked_results = sorted(
        zip(candidates, scores.tolist()),
        key=lambda x: x[1],
        reverse=True
    )

    return best_answer, ranked_results

In [49]:
test_questions = [
    "친구랑 싸웠어",
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "기분이 좋아"
]

for question in test_questions:
    best_answer, ranked_results = generate_answer_rerank(
        model,
        question,
        sp,
        device,
        reranker
    )

    print("질문:", question)
    print("최종 답변:", best_answer)
    print("후보 점수:")

    for answer, score in ranked_results[:5]:
        print(f"- {score:.4f} | {answer}")

    print("-" * 40)

/opt/conda/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


질문: 친구랑 싸웠어
최종 답변: 제가 들어드릴게요 .
후보 점수:
- 0.8017 | 제가 들어드릴게요 .
- 0.6804 | 마음 단단히 잡으세요 .
- 0.6589 | 마음에서도기가 좋죠 .
- 0.6578 | 먼저 용기내서 고백하는게 좋을 거예요 .
- 0.6385 | 마음 단단히 잡한 당신 .
----------------------------------------
질문: 오늘 너무 힘들어
최종 답변: 금방 지나갈 거예요 .
후보 점수:
- 0.7737 | 금방 지나갈 거예요 .
- 0.7382 | 이제 좀 괜찮아졌길 바랍니다 .
- 0.7272 | 이제 정말 안녕 .
- 0.7196 | 금방 나아질 거예요 .
- 0.6560 | 이제 좀 더 일찍 주무세요 .
----------------------------------------
질문: 여행 가고 싶어
최종 답변: 좋은 여행이에요 .
후보 점수:
- 0.7981 | 좋은 여행이에요 .
- 0.5334 | 먼저 다가가 보세요 .
- 0.5165 | 저랑 이야기해요 .
- 0.4853 | 그게 제일 쓸데 것 같아요 .
- 0.4765 | 좋은 생각만 사세요 .
----------------------------------------
질문: 내일 시험인데 걱정돼
최종 답변: 공부하러 왔나봐요 .
후보 점수:
- 0.8120 | 공부하러 왔나봐요 .
- 0.7531 | 처음이 될 거예요 .
- 0.7236 | 처음이 있는지 검색일 거예요 .
- 0.6766 | 처음이 있는지 검색겠어요 .
- 0.6453 | 다른 사람과 함께 있어아만 더 힘내세요 .
----------------------------------------
질문: 기분이 좋아
최종 답변: 제가 있잖아요 .
후보 점수:
- 0.9085 | 제가 있잖아요 .
- 0.7972 | 오랜만에 연락 받으면 반가 수 없어요 .
- 0.6860 | 오랜만에 시간 보내고이죠 .
- 0.5595 | 부모님이나 선생님께 도움을 청해보세요 .
- 0.3416 |

In [50]:
print("학습 질문을 벡터로 변환하는 중...")

question_embeddings = reranker.encode(
    questions,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64
)

print("질문 벡터 생성 완료!")
print("벡터 shape:", question_embeddings.shape)

학습 질문을 벡터로 변환하는 중...


Batches:   0%|          | 0/185 [00:00<?, ?it/s]

질문 벡터 생성 완료!
벡터 shape: torch.Size([11823, 384])


In [51]:
def retrieve_answer(
    sentence,
    questions,
    answers,
    question_embeddings,
    reranker,
    top_n=5
):
    sentence = preprocess_sentence(sentence)

    input_embedding = reranker.encode(
        sentence,
        convert_to_tensor=True
    )

    scores = util.cos_sim(
        input_embedding,
        question_embeddings
    )[0]

    top_results = torch.topk(scores, k=top_n)

    candidates = []

    for score, index in zip(
        top_results.values,
        top_results.indices
    ):
        idx = index.item()

        candidates.append({
            "score": score.item(),
            "question": questions[idx],
            "answer": answers[idx]
        })

    return candidates[0]["answer"], candidates

In [52]:
test_questions = [
    "친구랑 싸웠어",
    "오늘 너무 힘들어",
    "여행 가고 싶어",
    "내일 시험인데 걱정돼",
    "기분이 좋아"
]

for question in test_questions:
    best_answer, results = retrieve_answer(
        question,
        questions,
        answers,
        question_embeddings,
        reranker
    )

    print("입력 질문:", question)
    print("최종 답변:", best_answer)
    print("비슷한 학습 질문:")

    for result in results[:3]:
        print(
            f"- {result['score']:.4f} | "
            f"{result['question']} → {result['answer']}"
        )

    print("-" * 50)

입력 질문: 친구랑 싸웠어
최종 답변: 싸우면서 정 들 거예요 .
비슷한 학습 질문:
- 1.0000 | 친구랑 싸웠어 → 싸우면서 정 들 거예요 .
- 0.9731 | 친한 친구랑 싸웠어 → 친구라면 먼저 사과해보세요 .
- 0.9455 | 친구랑 말다툼했어 → 싸우면서 정 들 거예요 .
--------------------------------------------------
입력 질문: 오늘 너무 힘들어
최종 답변: 고생 많았어요 .
비슷한 학습 질문:
- 0.9903 | 오늘 너무 힘들다 → 고생 많았어요 .
- 0.9476 | 하루하루 너무 힘드네 → 다른사람에게 기대보는 건 어떨까요 .
- 0.9348 | 오늘도 힘들다 → 지금은 힘들겠지만 조금만 더 견뎌봐요 .
--------------------------------------------------
입력 질문: 여행 가고 싶어
최종 답변: 계획을 세워보세요 .
비슷한 학습 질문:
- 0.9765 | 여행가고 싶다 → 계획을 세워보세요 .
- 0.8887 | 여행이라도 가고싶다 → 이김에 떠나보세요 .
- 0.8799 | 여행 어디가지 → 편하게 쉴 수 있는 곳이 좋을 거예요 .
--------------------------------------------------
입력 질문: 내일 시험인데 걱정돼
최종 답변: 컨디션 조절 하세요 .
비슷한 학습 질문:
- 0.9439 | 내일 시험이야 → 컨디션 조절 하세요 .
- 0.9293 | 내일 발표인데 떨려 → 더 많이 연습하고 준비해보세요 .
- 0.9206 | 내일 발표 나는데 떨려 → 더 많이 연습하고 준비해보세요 .
--------------------------------------------------
입력 질문: 기분이 좋아
최종 답변: 저도 좋아요 .
비슷한 학습 질문:
- 0.9804 | 오늘 기분 좋아 → 저도 좋아요 .
- 0.9784 | 여기 있으니까 좋다 → 좋은 곳인가봐요 .
- 0.9775 | 느낌이 좋아 → 사랑의 느낌이

In [ ]:
입력으로 들어온 문장으로 우리가 갖고 있는 문장셋 > 가장 유사한 / 임베딩 유사도 간. 비슷한 학습 질문의 숫자 어떻게 매긴거니... / 확인해봐야 할 것 = 기존 가지고 있는 / 구현한 알고리즘과 최종답변 생성이 어떻게 된건지 알려달라. 유사도는 어떻게 계산한건지. 